<a href="https://colab.research.google.com/github/Shri2783/AQI-CHILID-HEALTH/blob/main/app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# app.py  —  AQI & Child Health India Dashboard
# Files needed in same folder: city_day.csv, NFHS_5_India_Districts_Factsheet_Data.csv
# Run locally:  streamlit run app.py
# ============================================================

!pip install streamlit
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
import os

st.set_page_config(
    page_title="AQI & Child Health India",
    page_icon="🌫️",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
.kpi{background:var(--background-color,#f8f9fa);border-radius:10px;
  padding:16px 10px;text-align:center;border:1px solid #e0e0e0}
.kpi-val{font-size:1.9rem;font-weight:700}
.kpi-lbl{font-size:0.78rem;color:#666;margin-top:2px}
.insight{border-left:4px solid;padding:12px 16px;margin:8px 0;
  font-size:0.9rem;line-height:1.6;border-radius:0 8px 8px 0}
.insight.red{border-color:#e74c3c;background:#fff5f5;color:#7b1f1f}
.insight.green{border-color:#27ae60;background:#f0fff4;color:#1a4731}
.insight.amber{border-color:#e67e22;background:#fffaf0;color:#6b3a0e}
.insight.blue{border-color:#2980b9;background:#eef6fd;color:#1a3a52}
.sol{border-radius:10px;padding:16px;margin-bottom:10px}
.sol.urgent{background:#fff0f0;border:1px solid #f5b7b1}
.sol.mid{background:#fffbf0;border:1px solid #fad7a0}
.sol.long{background:#f0fff4;border:1px solid #a9dfbf}
.discuss{background:#f0f4ff;border-radius:8px;padding:12px 16px;
  margin:8px 0;font-size:0.9rem;border-left:3px solid #3498db}
.badge{display:inline-block;padding:2px 9px;border-radius:12px;
  font-size:0.72rem;font-weight:600;margin-left:6px}
.bc{background:#e74c3c;color:white}
.bh{background:#e67e22;color:white}
.bm{background:#f1c40f;color:#333}
.bg{background:#27ae60;color:white}
</style>
""", unsafe_allow_html=True)

# ── Data paths ────────────────────────────────────────────────
BASE = '/content/' # os.path.dirname(os.path.abspath(__file__))
AQI_FILE  = os.path.join(BASE, "city_day.csv")
NFHS_FILE = os.path.join(BASE, "NFHS_5_India_Districts_Factsheet_Data.csv")

# ── Load & merge ──────────────────────────────────────────────
@st.cache_data
def load_data():
    city_state = {
        'Ahmedabad':'Gujarat','Aizawl':'Mizoram','Amaravati':'Maharashtra',
        'Amritsar':'Punjab','Bengaluru':'Karnataka','Bhopal':'Madhya Pradesh',
        'Brajrajnagar':'Odisha','Chandigarh':'Chandigarh','Chennai':'Tamil Nadu',
        'Coimbatore':'Tamil Nadu','Delhi':'Delhi','Ernakulam':'Kerala',
        'Gurugram':'Haryana','Guwahati':'Assam','Hyderabad':'Telangana',
        'Jaipur':'Rajasthan','Jorapokhar':'Jharkhand','Kochi':'Kerala',
        'Kolkata':'West Bengal','Lucknow':'Uttar Pradesh','Mumbai':'Maharashtra',
        'Patna':'Bihar','Shillong':'Meghalaya','Talcher':'Odisha',
        'Thiruvananthapuram':'Kerala','Visakhapatnam':'Andhra Pradesh',
    }
    aqi_raw = pd.read_csv(AQI_FILE)
    aqi_raw['Date']  = pd.to_datetime(aqi_raw['Date'], dayfirst=True, errors='coerce')
    aqi_raw['State'] = aqi_raw['City'].map(city_state)
    aqi_raw['Year']  = aqi_raw['Date'].dt.year
    aqi_full = aqi_raw.dropna(subset=['State'])
    aqi = aqi_full[aqi_full['Year'].between(2015,2019)].copy()

    aqi_state = aqi.groupby('State').agg(
        mean_AQI=('AQI','mean'), median_AQI=('AQI','median'),
        mean_PM25=('PM2.5','mean'), mean_PM10=('PM10','mean'),
        mean_NO2=('NO2','mean'), mean_SO2=('SO2','mean'),
        mean_O3=('O3','mean'), n_records=('AQI','count'),
    ).reset_index()

    def clean(c):
        return pd.to_numeric(
            c.astype(str).str.replace(r'[\(\)]','',regex=True).str.strip(),
            errors='coerce')

    nfhs = pd.read_csv(NFHS_FILE, encoding='latin1')
    nfhs.rename(columns={
        'State/UT':'State',
        'Children Prevalence of symptoms of acute respiratory infection (ARI) in the 2 weeks preceding the survey (Children under age 5 years) (%) ':'ARI_pct',
        'Children under 5 years who are stunted (height-for-age)18 (%)':'Stunting_pct',
        'Children under 5 years who are underweight (weight-for-age)18 (%)':'Underweight_pct',
        'Children age 6-59 months who are anaemic (<11.0 g/dl)22 (%)':'Anaemia_pct',
    }, inplace=True)
    for c in ['ARI_pct','Stunting_pct','Underweight_pct','Anaemia_pct']:
        nfhs[c] = clean(nfhs[c])
    nfhs_s = nfhs.groupby('State')[['ARI_pct','Stunting_pct',
                                     'Underweight_pct','Anaemia_pct']].mean().reset_index()

    for df in [aqi_state, nfhs_s]:
        df['State'] = df['State'].str.strip().str.title()

    m = pd.merge(aqi_state, nfhs_s, on='State', how='inner')

    def cat(v):
        if v<=50: return 'Good'
        elif v<=100: return 'Satisfactory'
        elif v<=200: return 'Moderate'
        elif v<=300: return 'Poor'
        else: return 'Very Poor'

    m['AQI_Category'] = m['mean_AQI'].apply(cat)
    m['AQI_Group']    = np.where(m['mean_AQI']>=m['mean_AQI'].median(),'High','Low')

    def g3(v):
        return 'Good / Satisfactory' if v<=100 else ('Moderate' if v<=200 else 'Poor / Very Poor')
    m['AQI_Group3'] = m['mean_AQI'].apply(g3)

    sc = MinMaxScaler()
    hc = ['ARI_pct','Stunting_pct','Underweight_pct','Anaemia_pct']
    m['Health_Index'] = sc.fit_transform(m[hc]).mean(axis=1)
    m['AQI_norm']     = sc.fit_transform(m[['mean_AQI']])

    am = m['AQI_norm'].median()
    hm = m['Health_Index'].median()
    def quad(r):
        ha = r['AQI_norm']    >= am
        hh = r['Health_Index'] >= hm
        if ha and hh:      return 'High Risk'
        elif not ha and hh: return 'Health Concern'
        elif ha and not hh: return 'Air Quality Concern'
        else:               return 'Favourable'
    m['Quadrant'] = m.apply(quad, axis=1)

    for hv in hc:
        m[hv+'_bin'] = (m[hv] >= m[hv].median()).astype(int)

    return m, aqi_full, aqi_raw

# ── Load ──────────────────────────────────────────────────────
try:
    merged, aqi_full, aqi_raw = load_data()
except FileNotFoundError as e:
    st.error(f"CSV file not found: {e}\n\nMake sure city_day.csv and "
             f"NFHS_5_India_Districts_Factsheet_Data.csv are in the same folder as app.py")
    st.stop()

# ── Constants ─────────────────────────────────────────────────
HV   = ['ARI_pct','Stunting_pct','Underweight_pct','Anaemia_pct']
HL   = {'ARI_pct':'ARI (%)','Stunting_pct':'Stunting (%)','Underweight_pct':'Underweight (%)','Anaemia_pct':'Anaemia (%)'}
PV   = ['mean_AQI','mean_PM25','mean_PM10','mean_NO2','mean_SO2','mean_O3']
CC   = {'Good':'#27ae60','Satisfactory':'#2ecc71','Moderate':'#f39c12','Poor':'#e67e22','Very Poor':'#e74c3c'}
QC   = {'High Risk':'#e74c3c','Health Concern':'#e67e22','Air Quality Concern':'#3498db','Favourable':'#27ae60'}
O3   = ['Good / Satisfactory','Moderate','Poor / Very Poor']

def sig(p):
    return '★★★' if p<0.001 else ('★★' if p<0.01 else ('★ p<0.05' if p<0.05 else 'ns'))

# ── Sidebar ───────────────────────────────────────────────────
with st.sidebar:
    st.markdown("## 🌫️ AQI & Child Health")
    st.markdown("*India — Non-Parametric Study*")
    st.caption("CPCB (2015–2019) + NFHS-5 (2019–21)")
    st.divider()
    page = st.radio("Navigate to", [
        "🏠  Overview",
        "📊  EDA Explorer",
        "🧪  Statistical Tests",
        "💡  Insights",
        "🛠️  Solutions",
        "🗣️  Discussion Guide",
    ])
    st.divider()
    st.caption("Shrigothay A · 22MIY0014")
    st.caption(f"States analysed: **{merged.shape[0]}**")
    st.caption(f"AQI records: **{len(aqi_full):,}**")

# ════════════════════════════════════════════════════════════
# PAGE 1 — OVERVIEW
# ════════════════════════════════════════════════════════════
if page == "🏠  Overview":
    st.title("🌫️ Urban Air Quality & Child Health in India")
    st.markdown("**Non-Parametric Analysis** | CPCB City-Day AQI + NFHS-5 District Data | 18 States")

    st.info("💡 Use the sidebar to explore data, run tests, read insights, and see evidence-based solutions.")

    c1,c2,c3,c4,c5 = st.columns(5)
    for col, val, lbl, color in zip(
        [c1,c2,c3,c4,c5],
        [merged.shape[0], aqi_full['City'].nunique(), f"{len(aqi_full):,}",
         (merged['Quadrant']=='High Risk').sum(), "4"],
        ["States analysed","Cities (CPCB)","AQI records","High-risk states","Tests applied"],
        ["#3498db","#27ae60","#9b59b6","#e74c3c","#e67e22"]
    ):
        col.markdown(f"""<div class="kpi">
        <div class="kpi-val" style="color:{color}">{val}</div>
        <div class="kpi-lbl">{lbl}</div></div>""", unsafe_allow_html=True)

    st.divider()
    col1, col2 = st.columns([3,2])

    with col1:
        st.subheader("State-wise Mean AQI (2015–2019)")
        fig = px.bar(
            merged.sort_values('mean_AQI', ascending=True),
            x='mean_AQI', y='State', color='AQI_Category',
            color_discrete_map=CC, orientation='h', height=560,
            labels={'mean_AQI':'Mean AQI','State':''},
        )
        fig.add_vline(x=100, line_dash='dash', line_color='red',
                      annotation_text='Moderate threshold (100)')
        fig.update_layout(plot_bgcolor='white', legend_title='Category',
                          margin=dict(l=0,r=10,t=10,b=0))
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        st.subheader("Study at a glance")
        st.markdown("""
**Why this study?**
India has some of the world's most polluted cities AND highest child malnutrition rates.
This study asks: are these two crises linked?

---
**What we found:**
- AQI significantly predicts **anaemia** (ρ = +0.550, p = 0.018)
- AQI significantly predicts **underweight** (ρ = +0.505, p = 0.033)
- High AQI states have **11% more anaemia** than Low AQI states
- **7 states** are HIGH RISK on both dimensions

---
**Why non-parametric?**
Only 18 states after merging. Non-parametric tests make no normality assumptions — they use ranks, making them ideal for small samples.
        """)

        st.subheader("Confirmed results")
        results = [
            ("AQI vs Anaemia", "ρ=+0.550", "p=0.018", "#27ae60"),
            ("AQI vs Underweight", "ρ=+0.505", "p=0.033", "#27ae60"),
            ("Mann–Whitney Anaemia", "r=−0.630", "p=0.027", "#27ae60"),
            ("Mann–Whitney Underweight", "r=−0.630", "p=0.027", "#27ae60"),
            ("Kruskal–Wallis Anaemia", "H=5.890", "p=0.053", "#e67e22"),
        ]
        for name, stat, pval, color in results:
            st.markdown(f"""<div style="display:flex;justify-content:space-between;align-items:center;
            padding:6px 10px;border-radius:6px;background:#f8f9fa;margin:3px 0;font-size:13px">
            <span>{name}</span>
            <span style="font-weight:600;color:{color}">{stat} · {pval}</span>
            </div>""", unsafe_allow_html=True)

# ════════════════════════════════════════════════════════════
# PAGE 2 — EDA EXPLORER
# ════════════════════════════════════════════════════════════
elif page == "📊  EDA Explorer":
    st.title("📊 EDA Explorer")
    st.caption("Explore the raw data — filter by city, year, and pollutant.")

    tab1, tab2, tab3, tab4 = st.tabs([
        "🏙️ City AQI Trends", "📍 State vs Health", "🔬 Pollutant Profiles", "📋 Summary Stats"
    ])

    with tab1:
        st.markdown("#### How does AQI change over time across cities?")
        c1, c2 = st.columns(2)
        with c1:
            cities = sorted(aqi_raw['City'].dropna().unique())
            sel = st.multiselect("Select cities", cities,
                                  default=['Delhi','Mumbai','Chennai','Kolkata','Patna'])
        with c2:
            yr = st.slider("Year range", 2015, 2024, (2015,2019))

        df_t = aqi_raw[aqi_raw['City'].isin(sel) &
                       aqi_raw['Year'].between(yr[0],yr[1])].dropna(subset=['AQI'])
        df_t['Month'] = df_t['Date'].dt.to_period('M').astype(str)
        monthly = df_t.groupby(['Month','City'])['AQI'].mean().reset_index()

        fig = px.line(monthly, x='Month', y='AQI', color='City', height=420,
                      title=f'Monthly mean AQI — {yr[0]} to {yr[1]}')
        fig.add_hline(y=100, line_dash='dash', line_color='orange', annotation_text='Moderate (100)')
        fig.add_hline(y=200, line_dash='dash', line_color='red', annotation_text='Poor (200)')
        fig.update_xaxes(tickangle=45)
        fig.update_layout(plot_bgcolor='white')
        st.plotly_chart(fig, use_container_width=True)

        st.markdown("""
        **What to notice:** Delhi and Patna spike sharply in Oct–Jan (winter crop burning + cold inversion).
        Southern cities (Chennai, Kochi) stay below 100 year-round.
        Children in North India face *months* of hazardous air every year.
        """)

    with tab2:
        st.markdown("#### Does higher AQI visually associate with worse child health?")
        c1, c2 = st.columns(2)
        with c1:
            xv = st.selectbox("X axis (pollutant)", PV)
        with c2:
            yv = st.selectbox("Y axis (health outcome)", HV, format_func=lambda x: HL[x])

        fig = px.scatter(
            merged, x=xv, y=yv, text='State', color='AQI_Category',
            size='n_records', trendline='ols',
            color_discrete_map=CC,
            hover_data=['State','mean_AQI','ARI_pct','Stunting_pct'],
            title=f'{xv} vs {HL[yv]}', height=500,
        )
        fig.update_traces(textposition='top center', marker_opacity=0.85)
        fig.update_layout(plot_bgcolor='white')
        st.plotly_chart(fig, use_container_width=True)
        st.markdown("""
        **What to look for:** A positive slope = higher pollution, worse health.
        States in the **top-right corner** = double burden — highest priority for intervention.
        """)

    with tab3:
        st.markdown("#### Which pollutant dominates in each state?")
        state_sel = st.multiselect(
            "Select states",
            sorted(merged['State'].tolist()),
            default=merged.sort_values('mean_AQI', ascending=False)['State'].head(6).tolist()
        )
        poll_map = {'mean_PM25':'PM2.5','mean_PM10':'PM10','mean_NO2':'NO₂','mean_SO2':'SO₂','mean_O3':'O₃'}
        df_p = merged[merged['State'].isin(state_sel)][['State']+list(poll_map.keys())].copy()
        df_p.columns = ['State'] + list(poll_map.values())
        df_melt = df_p.melt(id_vars='State', var_name='Pollutant', value_name='Concentration')
        fig = px.bar(df_melt, x='Pollutant', y='Concentration', color='State',
                     barmode='group', title='Pollutant concentrations by state', height=420)
        fig.update_layout(plot_bgcolor='white')
        st.plotly_chart(fig, use_container_width=True)
        st.markdown("""
        **Why it matters for solutions:** PM2.5 → industrial/burning.
        NO₂ → vehicle traffic. SO₂ → coal power plants.
        Knowing the dominant pollutant helps target the *right* intervention.
        """)

    with tab4:
        st.markdown("#### Descriptive statistics across 18 states")
        desc_cols = ['mean_AQI','mean_PM25','mean_PM10','ARI_pct',
                     'Stunting_pct','Anaemia_pct','Underweight_pct']
        st.dataframe(merged[desc_cols].describe().round(2), use_container_width=True)

        st.markdown("#### Full state data table")
        show_cols = ['State','AQI_Category','mean_AQI','ARI_pct',
                     'Stunting_pct','Anaemia_pct','Underweight_pct','Quadrant']
        st.dataframe(
            merged[show_cols].sort_values('mean_AQI', ascending=False).round(2),
            use_container_width=True
        )

# ════════════════════════════════════════════════════════════
# PAGE 3 — STATISTICAL TESTS
# ════════════════════════════════════════════════════════════
elif page == "🧪  Statistical Tests":
    st.title("🧪 Statistical Tests")
    st.caption("All 4 non-parametric tests — interactive with plain-language interpretation.")

    tab1, tab2, tab3, tab4 = st.tabs([
        "1️⃣ Spearman ρ", "2️⃣ Kruskal–Wallis", "3️⃣ Mann–Whitney U", "4️⃣ Chi-Square + V"
    ])

    with tab1:
        st.markdown("### Spearman rank correlation")
        st.markdown("Tests whether states that rank *higher on AQI* also tend to rank *higher on child health problems*.")

        poll_s = st.selectbox("Select pollutant", PV, key='sp')
        rows = []
        for hv in HV:
            d = merged[[poll_s, hv]].dropna()
            rho, p = stats.spearmanr(d[poll_s], d[hv])
            rows.append({
                'Health indicator': HL[hv], 'Spearman ρ': round(rho,3),
                'p-value': round(p,4), 'n': len(d), 'Result': sig(p)
            })
        df_res = pd.DataFrame(rows)

        def color_sig(val):
            if '★' in str(val): return 'background-color: #d4edda; color: #155724'
            if 'ns' in str(val): return 'color: #999'
            return ''

        st.dataframe(
            df_res.style.applymap(color_sig, subset=['Result']),
            use_container_width=True
        )
        st.caption("★★★ p<0.001 | ★★ p<0.01 | ★ p<0.05 | ns = not significant")

        st.markdown("#### Full heatmap — all pollutants vs all outcomes")
        rho_mat = {}
        for hv in HV:
            rho_mat[HL[hv]] = {}
            for pv in PV:
                d = merged[[pv,hv]].dropna()
                rho, _ = stats.spearmanr(d[pv], d[hv]) if len(d)>=3 else (np.nan, 1)
                rho_mat[HL[hv]][pv.replace('mean_','')] = round(rho,3)
        rho_df = pd.DataFrame(rho_mat).T
        fig = px.imshow(rho_df, text_auto=True, color_continuous_scale='RdYlGn',
                        zmin=-1, zmax=1, title="Spearman ρ heatmap", height=360)
        st.plotly_chart(fig, use_container_width=True)

        st.markdown("""
        <div class="insight blue">
        <b>How to read ρ:</b> +1 = perfect positive rank agreement | 0 = no relationship | −1 = inverse.
        ρ > 0.5 is a strong association. Both AQI vs Anaemia (ρ=0.550) and AQI vs Underweight (ρ=0.505)
        are statistically significant at p &lt; 0.05.
        </div>""", unsafe_allow_html=True)

    with tab2:
        st.markdown("### Kruskal–Wallis test")
        st.markdown("Tests whether child health *distributions* differ across 3 AQI groups.")

        hv2 = st.selectbox("Health outcome", HV, format_func=lambda x: HL[x], key='kw')
        groups = [merged[merged['AQI_Group3']==g][hv2].dropna() for g in O3]
        groups = [g for g in groups if len(g)>0]
        H, p = stats.kruskal(*groups)

        c1,c2,c3 = st.columns(3)
        c1.metric("H-statistic", f"{H:.3f}")
        c2.metric("p-value", f"{p:.4f}")
        c3.metric("Significance", sig(p))

        fig = px.violin(merged, x='AQI_Group3', y=hv2,
                        category_orders={'AQI_Group3':O3},
                        color='AQI_Group3', box=True, points='all',
                        color_discrete_sequence=['#27ae60','#f39c12','#e74c3c'],
                        hover_data=['State'],
                        title=f'{HL[hv2]} across AQI groups', height=420)
        fig.update_layout(plot_bgcolor='white', showlegend=False)
        st.plotly_chart(fig, use_container_width=True)

        st.markdown("#### All outcomes")
        kw_all = []
        for hv in HV:
            gs = [merged[merged['AQI_Group3']==g][hv].dropna() for g in O3]
            gs = [g for g in gs if len(g)>0]
            H2, p2 = stats.kruskal(*gs)
            kw_all.append({'Outcome': HL[hv], 'H': round(H2,3),
                           'p-value': round(p2,4), 'Result': sig(p2)})
        st.dataframe(pd.DataFrame(kw_all), use_container_width=True)

        st.markdown("""
        <div class="insight amber">
        <b>Why is Kruskal–Wallis mostly ns?</b> The Moderate AQI group contains 11 of 18 states,
        leaving only 3 + 4 states in the other groups. Small unequal groups reduce statistical power.
        The anaemia trend (p=0.053) is consistent with the significant Spearman result.
        </div>""", unsafe_allow_html=True)

    with tab3:
        st.markdown("### Mann–Whitney U test")
        st.markdown("Directly compares median child health between **High AQI** and **Low AQI** states.")

        med_aqi = merged['mean_AQI'].median()
        st.info(f"Median AQI split = **{med_aqi:.1f}** | "
                f"High AQI states: **{(merged['AQI_Group']=='High').sum()}** | "
                f"Low AQI states: **{(merged['AQI_Group']=='Low').sum()}**")

        mw_rows = []
        for hv in HV:
            h = merged[merged['AQI_Group']=='High'][hv].dropna()
            l = merged[merged['AQI_Group']=='Low'][hv].dropna()
            U, p = stats.mannwhitneyu(h, l, alternative='two-sided')
            r = round(1-(2*U)/(len(h)*len(l)), 3)
            diff = round(h.mean()-l.mean(), 2)
            mw_rows.append({
                'Outcome': HL[hv], 'U': round(U),
                'p-value': round(p,4), 'Effect r': r,
                'Mean High AQI': round(h.mean(),2),
                'Mean Low AQI': round(l.mean(),2),
                'Difference': f"+{diff}" if diff>0 else str(diff),
                'Result': sig(p)
            })
        st.dataframe(pd.DataFrame(mw_rows), use_container_width=True)

        hv3 = st.selectbox("Visualise", HV, format_func=lambda x: HL[x], key='mw3')
        fig = px.box(merged, x='AQI_Group', y=hv3, color='AQI_Group',
                     color_discrete_map={'High':'#e74c3c','Low':'#27ae60'},
                     points='all', hover_data=['State'],
                     title=f'{HL[hv3]}: High vs Low AQI states', height=420)
        fig.update_layout(plot_bgcolor='white', showlegend=False)
        st.plotly_chart(fig, use_container_width=True)

        st.markdown("""
        <div class="insight blue">
        <b>Effect size r = −0.630</b> for both anaemia and underweight — classified as <b>large</b>.
        High AQI states have 11.4% more anaemia and 7.8% more underweight children on average.
        These are clinically meaningful differences, not just statistical artefacts.
        </div>""", unsafe_allow_html=True)

    with tab4:
        st.markdown("### Chi-square + Cramér's V")
        st.markdown("Tests if AQI group *category* is associated with above/below-median child health outcomes.")

        chi_rows = []
        for hv in HV:
            ct = pd.crosstab(merged['AQI_Group'], merged[hv+'_bin'])
            chi2, p, dof, _ = stats.chi2_contingency(ct)
            n = ct.values.sum()
            v = round(np.sqrt(chi2/(n*(min(ct.shape)-1))), 3)
            effect = 'Large' if v>0.5 else ('Medium' if v>0.3 else 'Small')
            chi_rows.append({
                'Outcome': HL[hv], 'Chi²': round(chi2,3),
                'p-value': round(p,4), "Cramér's V": v,
                'Effect size': effect, 'Result': sig(p)
            })
        st.dataframe(pd.DataFrame(chi_rows), use_container_width=True)

        hv4 = st.selectbox("Visualise", HV, format_func=lambda x: HL[x], key='ch4')
        ct_n = pd.crosstab(merged['AQI_Group'], merged[hv4+'_bin'], normalize='index')*100
        ct_n.columns = ['Below median','Above median']
        ct_melt = ct_n.reset_index().melt(id_vars='AQI_Group',
                                           var_name='Outcome', value_name='Pct')
        fig = px.bar(ct_melt, x='AQI_Group', y='Pct', color='Outcome',
                     color_discrete_map={'Below median':'#27ae60','Above median':'#e74c3c'},
                     title=f'{HL[hv4]}: proportion above vs below median', height=380)
        fig.update_layout(plot_bgcolor='white')
        st.plotly_chart(fig, use_container_width=True)

        st.markdown("""
        <div class="insight amber">
        <b>Why is Chi-square ns despite Mann–Whitney being significant?</b>
        Binarising continuous data loses information. Cramér's V = 0.222 still shows
        a consistent small–medium effect across all nutritional outcomes.
        Chi-square needs larger n to reach significance with 2×2 tables.
        </div>""", unsafe_allow_html=True)

# ════════════════════════════════════════════════════════════
# PAGE 4 — INSIGHTS
# ════════════════════════════════════════════════════════════
elif page == "💡  Insights":
    st.title("💡 Insights — What the Data Tells Us")
    st.markdown("Plain-language interpretation of every result, connected to real-world meaning.")

    st.subheader("Risk quadrant map — 18 states")
    am = merged['AQI_norm'].median()
    hm = merged['Health_Index'].median()

    fig = px.scatter(
        merged, x='AQI_norm', y='Health_Index', text='State', color='Quadrant',
        color_discrete_map=QC, height=540,
        hover_data={'mean_AQI':':.1f','ARI_pct':':.1f',
                    'Stunting_pct':':.1f','Anaemia_pct':':.1f'},
        title='Quadrant: pollution burden vs child health burden',
        labels={'AQI_norm':'Normalised AQI (higher = worse)',
                'Health_Index':'Child health risk index (higher = worse)'},
    )
    fig.add_vline(x=am, line_dash='dash', line_color='gray', opacity=0.4)
    fig.add_hline(y=hm, line_dash='dash', line_color='gray', opacity=0.4)
    fig.update_traces(textposition='top center', marker=dict(size=14, opacity=0.85))
    fig.update_layout(plot_bgcolor='white')
    for txt, x, y, col in [
        ("HIGH RISK",0.88,0.95,'#e74c3c'),("Health concern",0.06,0.95,'#e67e22'),
        ("Air quality concern",0.80,0.05,'#3498db'),("Favourable",0.06,0.05,'#27ae60')
    ]:
        fig.add_annotation(x=x,y=y,text=txt,showarrow=False,
                           font=dict(size=10,color=col),xref='paper',yref='paper')
    st.plotly_chart(fig, use_container_width=True)

    st.divider()
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("🚨 HIGH RISK states")
        hr = merged[merged['Quadrant']=='High Risk'].sort_values('mean_AQI', ascending=False)
        for _, row in hr.iterrows():
            st.markdown(f"""<div class="insight red">
            <b>{row['State']}</b> — AQI: {row['mean_AQI']:.0f} |
            Anaemia: {row['Anaemia_pct']:.1f}% | Stunting: {row['Stunting_pct']:.1f}% |
            Underweight: {row['Underweight_pct']:.1f}%
            </div>""", unsafe_allow_html=True)

    with col2:
        st.subheader("✅ Favourable states")
        fav = merged[merged['Quadrant']=='Favourable'].sort_values('mean_AQI')
        for _, row in fav.iterrows():
            st.markdown(f"""<div class="insight green">
            <b>{row['State']}</b> — AQI: {row['mean_AQI']:.0f} |
            Anaemia: {row['Anaemia_pct']:.1f}% | Stunting: {row['Stunting_pct']:.1f}%
            </div>""", unsafe_allow_html=True)

    st.divider()
    st.subheader("Why does PM2.5 cause anaemia and underweight — not just ARI?")
    st.markdown("""
    <div class="insight blue">
    PM2.5 particles (&lt;2.5 µm) are small enough to cross the alveolar membrane into the bloodstream.
    Once in circulation they trigger chronic inflammation → release of IL-6 and TNF-α →
    upregulation of <b>hepcidin</b> → suppressed iron absorption → <b>anaemia</b>.
    The same inflammatory pathway disrupts growth hormone signalling → <b>stunting and underweight</b>.
    This is why AQI correlates with nutritional outcomes, not just respiratory disease.
    </div>""", unsafe_allow_html=True)

    st.subheader("Why is Kerala the benchmark?")
    kerala = merged[merged['State']=='Kerala'].iloc[0]
    st.markdown(f"""
    <div class="insight green">
    Kerala has the <b>lowest anaemia (38.3%)</b>, lowest ARI, and low AQI (77.9) among all matched states.
    Its success comes from high maternal education, strong ICDS coverage, clean cooking fuel adoption,
    and coastal geography reducing particulate accumulation.
    Replicating Kerala's integrated approach in HIGH RISK states is the long-term policy goal.
    </div>""", unsafe_allow_html=True)

# ════════════════════════════════════════════════════════════
# PAGE 5 — SOLUTIONS
# ════════════════════════════════════════════════════════════
elif page == "🛠️  Solutions":
    st.title("🛠️ Evidence-Based Solutions")
    st.markdown("Recommendations grounded in your data — not generic advice.")

    st.subheader("Solution framework by timeline")
    c1,c2,c3 = st.columns(3)

    with c1:
        st.markdown("""<div class="sol urgent">
        <h4 style="color:#a93226;margin:0 0 8px">🔴 Immediate (0–6 months)</h4>
        <ul style="margin:0;padding-left:18px;font-size:0.87rem;line-height:2">
        <li>AQI alerts to ASHA workers in Bihar, UP, Jharkhand</li>
        <li>Emergency iron/folic acid distribution in Gujarat (anaemia 79.8%), Rajasthan (72.2%)</li>
        <li>ARI surveillance in Oct–Jan high-pollution season</li>
        <li>Outdoor activity advisories for under-5 children on AQI &gt;200 days</li>
        </ul></div>""", unsafe_allow_html=True)

    with c2:
        st.markdown("""<div class="sol mid">
        <h4 style="color:#9a7d0a;margin:0 0 8px">🟡 Short-term (6–18 months)</h4>
        <ul style="margin:0;padding-left:18px;font-size:0.87rem;line-height:2">
        <li>Integrate AQI into district HMIS health reporting</li>
        <li>PM2.5 source attribution in Gujarat + UP</li>
        <li>Crop residue management in Punjab/Haryana with farmer compensation</li>
        <li>Train paediatric workers on pollution-related anaemia treatment</li>
        </ul></div>""", unsafe_allow_html=True)

    with c3:
        st.markdown("""<div class="sol long">
        <h4 style="color:#1e8449;margin:0 0 8px">🟢 Long-term (2–5 years)</h4>
        <ul style="margin:0;padding-left:18px;font-size:0.87rem;line-height:2">
        <li>100% LPG coverage in HIGH RISK states via PMUY expansion</li>
        <li>National AQI–Child Health surveillance dashboard</li>
        <li>Longitudinal cohort study across HIGH vs LOW AQI districts</li>
        <li>Revise NAAQS PM2.5 to WHO 2021 standard (15 µg/m³)</li>
        </ul></div>""", unsafe_allow_html=True)

    st.divider()
    st.subheader("State-level action plan")

    for _, row in merged.sort_values('Health_Index', ascending=False).iterrows():
        badge_map = {
            'High Risk': '<span class="badge bc">Critical</span>',
            'Health Concern': '<span class="badge bh">Health</span>',
            'Air Quality Concern': '<span class="badge bm">Air</span>',
            'Favourable': '<span class="badge bg">OK</span>',
        }
        badge = badge_map.get(row['Quadrant'],'')
        with st.expander(f"{row['State']}  —  AQI: {row['mean_AQI']:.0f}  {badge}",
                         unsafe_allow_html=True):
            c1,c2 = st.columns(2)
            with c1:
                st.markdown(f"""
                **AQI Category:** {row['AQI_Category']}
                **ARI:** {row['ARI_pct']:.1f}%
                **Stunting:** {row['Stunting_pct']:.1f}%
                **Anaemia:** {row['Anaemia_pct']:.1f}%
                **Underweight:** {row['Underweight_pct']:.1f}%
                **Quadrant:** {row['Quadrant']}
                """)
            with c2:
                dominant = merged[['mean_PM25','mean_PM10','mean_NO2','mean_SO2']].loc[row.name].idxmax()
                dominant = dominant.replace('mean_','')
                st.markdown(f"**Dominant pollutant:** {dominant}")
                actions = {
                    'High Risk': "- Immediate IFA supplementation\n- ASHA AQI health alert system\n- Source control for dominant pollutant\n- ARI sentinel surveillance",
                    'Health Concern': "- ICDS nutrition program review\n- Expand NFHS monitoring\n- Clean cooking fuel push",
                    'Air Quality Concern': "- Preventive child health screening\n- Industrial emission controls\n- Monitor before health damage occurs",
                    'Favourable': "- Document what is working\n- Share best practices with HIGH RISK states\n- Continue NFHS monitoring"
                }
                st.markdown(actions.get(row['Quadrant'],''))

    st.divider()
    st.subheader("Pollutant → health pathway")
    pathway_data = {
        'Pollutant': ['PM2.5','PM10','NO₂','SO₂','O₃'],
        'Primary pathway': [
            'Enters bloodstream → hepcidin → iron suppression',
            'Upper respiratory tract irritation',
            'Airway inflammation → lung development impairment',
            'Mucous membrane irritation',
            'Oxidative stress → growth interference'
        ],
        'Child outcome': ['Anaemia, underweight, stunting','ARI','ARI, stunting','ARI','Underweight, stunting'],
        'Most affected state': ['Gujarat (AQI 484)','UP (AQI 224)','Haryana (AQI 235)','Jharkhand','Rajasthan']
    }
    st.dataframe(pd.DataFrame(pathway_data), use_container_width=True)

# ════════════════════════════════════════════════════════════
# PAGE 6 — DISCUSSION GUIDE
# ════════════════════════════════════════════════════════════
elif page == "🗣️  Discussion Guide":
    st.title("🗣️ Discussion Guide")
    st.markdown("Prepare for your viva, presentation, or project review with these ready answers.")

    tab1, tab2, tab3 = st.tabs(["❓ Q&A", "📣 Talking Points", "📝 Limitations"])

    with tab1:
        qa = [
            ("Why non-parametric tests instead of regression?",
             "After merging CPCB and NFHS-5 we had only 18 state-level observations. OLS regression with 18 points and multiple predictors is underpowered and violates normality assumptions. Non-parametric tests use ranks — not raw values — making them robust for small samples and skewed distributions like AQI (range 44–484)."),
            ("Correlation isn't causation — how do you address this?",
             "Correct. This is a cross-sectional ecological study, so we cannot prove causation. However, the association is consistent across 4 independent tests, biologically plausible (PM2.5 triggers hepcidin-mediated iron suppression), and aligns with existing literature. The study's goal is to identify priority states for intervention, not establish causation — that requires a longitudinal cohort design."),
            ("Why only 18 states — why not all 36?",
             "CPCB monitors only 26 cities in 20 distinct states. After name harmonisation between CPCB and NFHS-5, 18 states matched cleanly. States without CPCB monitoring cannot be included. This coverage gap is itself a finding — monitoring is concentrated in urban/large states, creating blind spots in rural and North-East India."),
            ("Why did ARI show no correlation (ρ=+0.046)?",
             "Three reasons: (1) NFHS-5 ARI is a 2-week recall measure subject to seasonal variation not captured in 5-year mean AQI. (2) ARI is strongly influenced by vaccination coverage and healthcare access, which may overwhelm the pollution signal. (3) The ARI range is narrow (0.3–4.3%), limiting variance available for correlation."),
            ("What is Cramér's V and why use it alongside Chi-square?",
             "Chi-square tells you whether an association is statistically significant but is sensitive to sample size. Cramér's V is a normalised effect size (0 to 1) that tells you how strong the association is, independent of n. V=0.222 for nutritional outcomes indicates a consistent small-medium practical association even where p>0.05 due to small n."),
            ("Gujarat has the highest AQI (484) but relatively low ARI — why?",
             "ARI is primarily driven by PM10 and NO₂ from respiratory irritants, while Gujarat's extreme AQI is largely from industrial PM2.5 which preferentially causes systemic effects (anaemia, underweight) rather than acute respiratory infection. Also, Gujarat has better healthcare infrastructure than Bihar/UP, potentially reducing reported ARI rates."),
        ]
        for q, a in qa:
            st.markdown(f'<div class="discuss"><b>Q: {q}</b></div>', unsafe_allow_html=True)
            with st.expander("See answer"):
                st.markdown(a)

    with tab2:
        points = [
            ("The problem is real and measurable",
             "India has some of the world's highest PM2.5 levels AND the world's highest child stunting burden. This study is among the first to quantify the link using nationally representative data (NFHS-5) and government monitoring data (CPCB) simultaneously."),
            ("The Indo-Gangetic Plain is a crisis zone",
             "Bihar (AQI 252, stunting 42.6%), UP (AQI 224, anaemia 66.4%), and West Bengal (AQI 148, anaemia 69.8%) cluster in the HIGH RISK quadrant. These 3 states alone hold ~350 million people. Targeted intervention here has the highest potential impact per rupee spent."),
            ("Effect sizes are large — this is not trivial",
             "Mann–Whitney effect size r=−0.630 for both anaemia and underweight is classified as LARGE by Cohen's benchmarks. High AQI states have 11.4 percentage points more anaemia than Low AQI states. That's not noise — it's a clinically meaningful gap."),
            ("Non-parametric was the correct methodological choice",
             "With n=18 and AQI ranging from 44 to 484, parametric assumptions cannot be met. Spearman and Mann–Whitney give valid inference where Pearson and t-tests would not. This is methodological rigour, not a limitation."),
            ("Kerala shows what good looks like",
             "Lowest anaemia (38.3%), lowest stunting (23.3%), AQI below 100 — Kerala demonstrates that low pollution and good child health co-occur and can be achieved through integrated policy. It's a proof of concept for what HIGH RISK states should target."),
        ]
        for title, body in points:
            st.markdown(f"""<div class="insight blue">
            <b>{title}</b><br>{body}</div>""", unsafe_allow_html=True)

    with tab3:
        st.subheader("Honest limitations")
        lims = {
            "Small n (18 states)": "Limits power for Kruskal–Wallis and Chi-square. Many moderate effects don't reach significance not because they're absent but because n is too small.",
            "Ecological fallacy": "State-level aggregation masks within-state variation. A district-level analysis would be more precise.",
            "Temporal mismatch": "CPCB 2015–2019 vs NFHS-5 2019–2021. Some pollution-health lag may exist.",
            "Urban monitoring only": "CPCB stations are urban. Rural biomass burning pollution is not captured, potentially underestimating exposure in HIGH RISK states.",
            "Uncontrolled confounders": "Poverty, sanitation, maternal education all affect child health and may correlate with AQI. Without controlling for these, residual confounding is possible.",
            "Cross-sectional design": "Cannot establish causal direction. Reverse causality (poverty → both pollution and poor health) cannot be ruled out."
        }
        for lim, desc in lims.items():
            with st.expander(lim):
                st.markdown(desc)

        st.subheader("Future work")
        st.markdown("""
        1. **Satellite PM2.5** (NASA MERRA-2 / Sentinel-5P) to extend to all 707 districts
        2. **Longitudinal analysis** — compare NFHS-4 (2015–16) vs NFHS-5 (2019–21) to see if pollution improvements track health improvements
        3. **Multilevel modelling** — nest districts within states to handle hierarchical structure
        4. **Machine learning** — Random Forest to rank which factors best predict child health
        5. **Cost-benefit analysis** — estimate DALYs averted if HIGH RISK states reach AQI &lt; 100
        """)

2026-04-13 06:58:45.876 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:58:45.878 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:58:45.879 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:58:45.881 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:58:45.886 No runtime found, using MemoryCacheStorageManager
2026-04-13 06:58:45.918 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:58:45.919 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:58:45.920 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:58:45.920 Thread 'MainThread':